In [ ]:
# Cell 1: Clone code (Kaggle: bật Internet trong Settings). Nhánh phải đã push code mới nhất
!git clone --single-branch --branch HoangMinh https://github.com/VanHung-05/zalo_challenge_2026_datamate.git
%cd zalo_challenge_2026_datamate
!git log --oneline -1

In [ ]:
# Cell 2: Cài thư viện, trừ llama-cpp-python (cài bản CUDA ở cell sau, tránh build bản CPU vô ích)
!grep -v '^llama-cpp-python' requirements.txt > /tmp/requirements_no_llama.txt
!pip install -q -r /tmp/requirements_no_llama.txt
!pip install -q diskcache jinja2  # dependencies của llama-cpp-python (cell sau dùng --no-deps)

In [ ]:
# Cell 2b: Cài llama-cpp-python bản CUDA (compile từ source, ~10-15 phút, chạy 1 lần)
# --no-deps: không cài lại numpy... của Kaggle; CMAKE_CUDA_ARCHITECTURES=native: chỉ build cho GPU đang có (nhanh hơn)
# runner chạy bằng !python (tiến trình mới) nên KHÔNG cần restart kernel
import shutil
if shutil.which("nvidia-smi"):
    !nvidia-smi --query-gpu=name,memory.total --format=csv
    !CMAKE_ARGS="-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=native" pip install llama-cpp-python --force-reinstall --no-deps --no-cache-dir -q
else:
    print("ℹ️ Không có GPU — cài bản CPU")
    !pip install -q llama-cpp-python
!python -c "import llama_cpp; print('llama-cpp-python', llama_cpp.__version__, '| GPU offload:', llama_cpp.llama_supports_gpu_offload())"

## Download model

In [ ]:
!python scripts/download_model.py

## Test inference

In [ ]:
# Chạy thử 5 câu train; xem thời gian/câu (elapsed_seconds) và đáp án trước khi chạy cả bộ
!python src/datamate/runner.py -c configs/dev.yaml -d train --limit 5 -o outputs/train_5/submission.jsonl
!python scripts/evaluate.py -p outputs/train_5/submission.jsonl -g train/answers.jsonl --only-predicted

In [ ]:
# Xem log 1 câu: SQL model sinh ra, lỗi, kết quả
import glob, json
log = json.load(open(sorted(glob.glob('outputs/train_5/logs/*.json'))[0], encoding='utf-8'))
print(log['question'], '->', log['final'], f"({log['elapsed_seconds']}s)")
for a in log.get('attempts', []):
    print('--- attempt', a['attempt'], '| error:', a.get('error'))
    print(a.get('sql'))

## Inference public test

In [ ]:
# Chạy toàn bộ public test (1000 câu), ghi từng dòng nên dừng giữa chừng vẫn giữ kết quả đã chạy
!python src/datamate/runner.py -c configs/dev.yaml -d public_test -o outputs/public/submission.jsonl
!python scripts/validate_submission.py -p outputs/public/submission.jsonl -q public_test/questions.jsonl

In [ ]:
# Copy file nộp ra /kaggle/working để tải về từ tab Output
!cp outputs/public/submission.jsonl /kaggle/working/submission.jsonl && ls -la /kaggle/working/submission.jsonl